In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
from scripts.helpers import (
    data_split,
    fairness_summary_by_group,
    generate_embeddings,
    plot_feature_importance,
    run_fairness_checks,
    save_classification_report,
    top_k_feature_names,
)

RANDOM_STATE = 42

In [16]:
df = pd.read_csv(PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv")


In [17]:
import numpy as np

print("Applying Cyclical Encoding to Time Features...")

# 1. Clean and convert arrival_time (HHMM integer) to a continuous hour scale (0 to 23.99)
# Example: 1430 -> 14 hours + (30/60) minutes = 14.5
if 'arrival_time' in df.columns:
    # Set missing or invalid times (often negative in NHAMCS) to NaN    
    df['arrival_time'] = np.where(df['arrival_time'] < 0, np.nan, df['arrival_time'])
    
    # Extract hours and minutes
    hours = np.floor_divide(df['arrival_time'], 100)
    minutes = np.mod(df['arrival_time'], 100)
    
    # Create the continuous hour feature
    df['arrival_hour_float'] = hours + (minutes / 60.0)

# 2. Define the exact maximum values for a full cycle
cycle_maxes = {
    'visit_month': 12.0,
    'day_of_week': 7.0,
    'arrival_hour_float': 24.0
}

# 3. Apply the Sine and Cosine Transformations
for col, max_val in cycle_maxes.items():
    if col in df.columns:
        # Calculate the angle on the circle
        angle = 2 * np.pi * df[col] / max_val
        
        # Create the Sin and Cos features
        df[f'{col}_sin'] = np.sin(angle)
        df[f'{col}_cos'] = np.cos(angle)

# 4. Drop the original linear time columns to prevent multicollinearity
cols_to_drop = ['visit_month', 'day_of_week', 'arrival_time', 'arrival_hour_float']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Dataframe shape after cyclical encoding: {df.shape}")
print("\nSample of the new Time Features:")
display(df[[c for c in df.columns if '_sin' in c or '_cos' in c]].head())

Applying Cyclical Encoding to Time Features...
Dataframe shape after cyclical encoding: (58124, 52)

Sample of the new Time Features:


,visit_month_sin,visit_month_cos,day_of_week_sin,day_of_week_cos,arrival_hour_float_sin,arrival_hour_float_cos
0,-2.449294e-16,1.000000,0.781831,0.623490,-0.748956,0.662620
1,-2.449294e-16,1.000000,0.781831,0.623490,-0.957571,0.288196
2,-2.449294e-16,1.000000,-0.781831,0.623490,-0.625923,-0.779884
3,-2.449294e-16,1.000000,-0.433884,-0.900969,-0.994522,-0.104528
4,-5.000000e-01,0.866025,0.974928,-0.222521,-0.983255,-0.182236


In [18]:
# --- NLP EMBEDDINGS (WITH CACHING) ---
print("NLP Embedding Logic...")

df_text = df[["chief_complaint_text", "injury_cause_text"]].fillna("")
df = df.drop(columns=["chief_complaint_text", "injury_cause_text"])
df_emb = generate_embeddings(df_text, filename=PROJECT_ROOT / "working_data" / "embeddings.parquet")

NLP Embedding Logic...
🚀 Loading cached embeddings from /home/gaurav/python/kaggle/triage/working_data/embeddings.parquet...


In [19]:
print("Adding clinical ratios (Shock Index, MAP)...")

df["shock_index"] = df["heart_rate"] / df["sys_bp"].replace(0, np.nan)

df["map"] = (df["sys_bp"] + 2 * df["dias_bp"]) / 3

df["pulse_pressure"] = df["sys_bp"] - df["dias_bp"]

df["age_hr_interaction"] = df["age"] * df["heart_rate"]

df["resp_spo2_ratio"] = df["resp_rate"] / df["spo2"].replace(0, np.nan)

history_cols = [c for c in df.columns if any(k in c for k in "hist_")]
if history_cols:
    hist_numeric = df[history_cols].apply(pd.to_numeric, errors="coerce")
    df["history_count"] = hist_numeric.fillna(0).sum(axis=1)

Adding clinical ratios (Shock Index, MAP)...


In [20]:
df = pd.concat([df, df_emb], axis=1)
# 1. Define your new 'Blind' Feature Set
features_to_remove = ['insurance', 'residence', 'no_payment', 'region', 'history_count']
df_blind = df.drop(columns=[c for c in features_to_remove if c in df.columns])
x_train, x_val, y_train, y_val = data_split(df_blind, config="base_line", random_state=RANDOM_STATE)

In [ ]:
from scripts.models import train_xgboost


weighted_model = train_xgboost(x_train, y_train, config="xgb_big")
y_pred = weighted_model.predict(x_val)
save_classification_report(
    y_val,
    y_pred,
    model_name="xgb_weighted_nlp",
    seed=RANDOM_STATE,
    config="xgb_weighted",
    columns=x_train.columns,
    notes="weighted XGB with NLP embeddings",
)
plot_feature_importance(
    weighted_model,
    x_train.columns,
    top_n=20,
    output_path=PROJECT_ROOT / "fig" / "feature_importance_big_nlp.png",
)

In [ ]:
# Feature importance list (sorted) and optional top-k selection
import pandas as pd

importances = weighted_model.feature_importances_
fi_df = pd.DataFrame({
    "feature": x_train.columns,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(fi_df.head(130))


fi_path = PROJECT_ROOT / "results" / "feature_importance" / "xgb_weighted_nlp_feature_importance.csv"
fi_path.parent.mkdir(parents=True, exist_ok=True)
fi_df.to_csv(fi_path, index=False)

# Optional: keep only top-k features for a follow-up model
TOP_K = 50
top_features = fi_df.head(TOP_K)["feature"].tolist()
x_train_top = x_train[top_features]
x_val_top = x_val[top_features]
print(f"Top-K features saved. x_train_top shape: {x_train_top.shape}")

                   feature  importance
0              ems_arrival    0.077495
1               no_payment    0.041818
2    chief_complaint_emb_2    0.023917
3              hist_stroke    0.016377
4    chief_complaint_emb_4    0.015629
5            history_count    0.015133
6          visit_month_sin    0.015104
7                   region    0.013736
8    chief_complaint_emb_0    0.013520
9                 hist_ckd    0.013385
10   chief_complaint_emb_6    0.012972
11             hist_cancer    0.012971
12                hist_cad    0.012897
13                hist_hiv    0.012745
14                 episode    0.012711
15                 hist_pe    0.012666
16      age_hr_interaction    0.012607
17         visit_month_cos    0.011686
18              pain_score    0.011638
19   chief_complaint_emb_1    0.011457
20      injury_cause_emb_0    0.011374
21     injury_cause_emb_12    0.011176
22   chief_complaint_emb_5    0.010996
23       hist_hypertension    0.010716
24        hist_diabetes_t

In [ ]:
from sklearn.feature_selection import SelectFromModel

# Fit selector using the trained weighted model
selector = SelectFromModel(weighted_model, threshold="0.1*mean", prefit=True)

# Selected feature mask and names
selected_feature_names = x_train.columns[selector.get_support()].tolist()
print(f"Reduced from {x_train.shape[1]} to {len(selected_feature_names)} features.")

Reduced from 97 to 97 features.


In [ ]:
# Keep original dtypes by slicing columns directly
X_train_reduced = x_train[selected_feature_names].copy()
X_val_reduced = x_val[selected_feature_names].copy()

print(f"Selected features: {len(selected_feature_names)}")

Selected features: 97


In [ ]:
pruned_model = train_xgboost(X_train_reduced, y_train, config="xgb_weighted")
y_pred = pruned_model.predict(X_val_reduced)
save_classification_report(
    y_val,
    y_pred,
    model_name="xgb_weighted_nlp_pruned",
    seed=RANDOM_STATE,
    config="xgb_weighted",
    columns=selected_feature_names,

    notes="weighted XGB with NLP embeddings pruned",
)

              precision    recall  f1-score   support

           0       0.24      0.18      0.21       178
           1       0.38      0.55      0.45      1706
           2       0.70      0.51      0.59      5895
           3       0.53      0.62      0.57      3358
           4       0.20      0.31      0.25       488

    accuracy                           0.54     11625
   macro avg       0.41      0.43      0.41     11625
weighted avg       0.58      0.54      0.54     11625



{'0': {'precision': 0.23880597014925373,
  'recall': 0.1797752808988764,
  'f1-score': 0.20512820512820512,
  'support': 178.0},
 '1': {'precision': 0.3788492706645057,
  'recall': 0.5480656506447831,
  'f1-score': 0.44801149976042165,
  'support': 1706.0},
 '2': {'precision': 0.7036950964443411,
  'recall': 0.5136556403731977,
  'f1-score': 0.5938419297901549,
  'support': 5895.0},
 '3': {'precision': 0.5255474452554745,
  'recall': 0.6217986896962477,
  'f1-score': 0.5696357932069295,
  'support': 3358.0},
 '4': {'precision': 0.2034805890227577,
  'recall': 0.3114754098360656,
  'f1-score': 0.24615384615384617,
  'support': 488.0},
 'accuracy': 0.5363440860215054,
 'macro avg': {'precision': 0.41007567430726655,
  'recall': 0.4349541342898341,
  'f1-score': 0.41255425480791147,
  'support': 11625.0},
 'weighted avg': {'precision': 0.5764467751045672,
  'recall': 0.5363440860215054,
  'f1-score': 0.544901392320775,
  'support': 11625.0}}